# 8. 선택 실습: 실제 모델로 교체하기

근거: 설계 아이디어를 LangChain 모델 adapter로 연결하는 교육용 확장. [원문](https://x.com/DeepSeekHarness/status/2116857257245987236?s=20).

**실행:** Python (Harness Lab) 커널에서 위에서 아래로 실행하세요. 각 노트북은 새 커널에서 독립 실행됩니다. 기본 셀은 외부 API를 호출하지 않습니다.

## 결정적 실습과 실제 모델 평가를 분리
앞선 과정은 구조를 검증했습니다. 의미 판단 능력, tool 선택 정확도, 지연·비용은 실제 모델로 별도 평가해야 합니다. 아래 셀은 기본적으로 네트워크 요청을 하지 않습니다.

환경에 DEEPSEEK_API_KEY, DEEPSEEK_MODEL을 안전하게 등록하고 api.deepseek.com 접근을 허용한 뒤 RUN_LIVE=1을 명시해야 실행합니다. 키를 노트북이나 출력에 붙여 넣지 마세요. 모델 이름은 계정에서 현재 지원하는 것을 사용합니다. 이 자료 제작 시 실제 모델 호출은 실행하지 않았습니다.

OpenAI 호환 chat API를 LangChain ChatOpenAI로 호출합니다. API 호환 세부사항과 모델 기능은 서비스 버전에 따라 확인해야 합니다. 키의 존재만으로 인증 성공을 판정하지 않습니다.

In [1]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
if os.getenv('RUN_LIVE')=='1':
    if not os.getenv('DEEPSEEK_API_KEY') or not os.getenv('DEEPSEEK_MODEL'):
        raise RuntimeError('환경 설정에 DEEPSEEK_API_KEY와 DEEPSEEK_MODEL이 필요합니다')
    model=ChatOpenAI(model=os.environ['DEEPSEEK_MODEL'],
        api_key=os.environ['DEEPSEEK_API_KEY'],base_url='https://api.deepseek.com',
        timeout=30,max_retries=0)
    prompt=ChatPromptTemplate.from_messages([
        ('system','Router의 책임만 한국어 두 문장으로 설명하세요. 작업을 실행하지 마세요.'),
        ('human','{question}')])
    chain=prompt | model | StrOutputParser()
    answer=chain.invoke({'question':'Router와 Worker를 분리하는 이유는?'})
    assert answer.strip()
    print(answer)
else:
    print('SKIPPED: 실제 모델 호출은 RUN_LIVE=1에서만 실행합니다.')

SKIPPED: 실제 모델 호출은 RUN_LIVE=1에서만 실행합니다.


## 평가 설계
01의 RunnableLambda를 바꾸려면 chat model에 bind_tools([lookup])를 적용하고 AIMessage.tool_calls를 기존 graph가 처리하도록 연결합니다. 02의 semantic 정책은 단순 자유 텍스트보다 검증 가능한 분류 schema가 적합합니다. 분류 결과에 권한을 부여하지 말고 코드 정책을 별도로 유지하세요.

평가셋에 잡담, 명시 요청, 후속 취소, 같은 Thread의 새 과제, 중복 사건을 포함하세요. 처리율뿐 아니라 잘못 시작한 작업·놓친 취소·Router 호출 수·p95 응답 시간·token 사용량을 기록합니다. 원문 예비 실험 수치를 자신의 환경 성능으로 인용하지 마세요.

## 직접 바꿔 보기

의미 필터가 틀린 경우 두 종류(false positive/negative)를 정의하고 각각 세 사례씩 작성하세요.

## 해설과 참고 답안

잡담을 작업으로 시작하면 false positive, 실제 오류 보고나 취소를 무시하면 false negative입니다. 단순 정확도 대신 사건 유형별 precision/recall과 사용자 개입량을 함께 비교하세요.